<a href="https://colab.research.google.com/github/vikramvundyala/python_AI-ML/blob/main/rag_customer_support_vikram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community langchain-openai faiss-cpu datasets


In [7]:
from datasets import load_dataset
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough


class CustomerSupportRagSystem:
    def __init__(self):
        self.embedder = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2"
        )
        self.vector_db = None
        self.retriever = None
        self.test_data = None
        self.lc_docs = None

    # -----------------------------
    # Load Dataset
    # -----------------------------
    def loadData(self):
        self.test_data = load_dataset("galileo-ai/ragbench", "delucionqa", split="test")
        print(self.test_data.column_names)

        documents = [row["documents"] for row in self.test_data]
        all_doc_strings = [doc_str for doc_list in documents for doc_str in doc_list]
        self.lc_docs = [Document(page_content=doc_str) for doc_str in all_doc_strings]

        return self.lc_docs

    # -----------------------------
    # Save FAISS
    # -----------------------------
    def saveToVectorDb(self):
        # CREATE NEW VECTOR DB IF DOES NOT EXIST
        self.vector_db = FAISS.from_documents(self.lc_docs, self.embedder)
        self.vector_db.save_local("faiss_index")
        self.retriever = self.vector_db.as_retriever(search_k=5)

    # -----------------------------
    # Load FAISS
    # -----------------------------
    def loadFromVectorDb(self):
        self.vector_db = FAISS.load_local(
            "faiss_index",
            self.embedder,
            allow_dangerous_deserialization=True
        )
        self.retriever = self.vector_db.as_retriever(search_k=5)

    # -----------------------------
    # LLM
    # -----------------------------
    def set_llm(self):
        load_dotenv()
        return ChatOpenAI(
            model_name="gpt-3.5-turbo",
            temperature=0.5,
            api_key=os.getenv("OPENAI_API_KEY"),
        )

    # -----------------------------
    # Format docs
    # -----------------------------
    def format_docs(self, docs: list[Document]) -> str:
        return "\n\n".join(doc.page_content for doc in docs)

    # -----------------------------
    # RAG Chain
    # -----------------------------
    def setup_rag_chain(self):
        template = """Answer the question based ONLY on the following context:

                    {context}

                    Question: {question}
                    """
        prompt = ChatPromptTemplate.from_template(template)

        rag_chain = (
            RunnableParallel(
                context=self.retriever,        # retrieved docs
                question=RunnablePassthrough()  # original question
            )
            | RunnableParallel(
                result=(
                    RunnablePassthrough.assign(
                        context=lambda x: self.format_docs(x["context"])
                    )
                    | prompt
                    | self.set_llm()
                    | StrOutputParser()
                ),
                source_documents=lambda x: x["context"]
            )
        )
        return rag_chain

    # -----------------------------
    # MAIN PIPELINE
    # -----------------------------
    def main(self):

        print(" Loading dataset...")
        self.loadData()

        # Create if not exists, else load
        if os.path.exists("faiss_index"):
            print(" Loading existing FAISS DB...")
            self.loadFromVectorDb()
        else:
            print(" Creating new FAISS DB...")
            self.saveToVectorDb()

        query = self.test_data[0]["question"]
        expected_answer = self.test_data[0]["response"]

        rag_chain = self.setup_rag_chain()
        response = rag_chain.invoke(query)

        print("\n====================================")
        print(" QUESTION:", query)
        print(" MODEL ANSWER:", response["result"])
        print(" TRUE ANSWER:", expected_answer)
        print(" SOURCES:", response["source_documents"])
        print("====================================\n")

In [10]:
rag = CustomerSupportRagSystem()
rag.main()

 Loading dataset...
['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score']
 Loading existing FAISS DB...

 QUESTION: What if I fail to latch the tailgate properly?
 MODEL ANSWER: Failure to securely latch the tailgate could result in damage to the vehicle or cargo.
 TRUE ANSWER: If you fail to securely latch the tailgate properly, it could result in damage to the vehicle or cargo.
 SOURCES: [Document(id='bfc237e7-420b-44de-9dc2-616648db1a44', metadata={}, page

In [9]:
!zip -r faiss_index.zip faiss_index/

  adding: faiss_index/ (stored 0%)
  adding: faiss_index/index.faiss (deflated 11%)
  adding: faiss_index/index.pkl (deflated 76%)


In [ ]:
!unzip my_archive.zip